In [10]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [11]:
load_dotenv()

True

In [12]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [13]:
#structured data schema using pydantic to control llm output

class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [14]:
structured_llm = llm.with_structured_output(EvaluationSchema)

In [15]:
essay = """
hello this is the best essay in the world.
"""

In [17]:
class essayState(TypedDict):                        #define the state of the workflow
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float
        

In [18]:
def evaluate_language(state: essayState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_llm.invoke(prompt)

    return {'language_feedback': output.feedback,
            'individual_scores': [output.score]}

In [19]:
def evaluate_analysis(state: essayState):
    prompt = f'Evaluate the analytical quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_llm.invoke(prompt)

    return {'analysis_feedback': output.feedback,
            'individual_scores': [output.score]}

In [20]:
def evaluate_thought(state: essayState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_llm.invoke(prompt)

    return {'clarity_feedback': output.feedback,
            'individual_scores': [output.score]}

In [21]:
def final_evaluation(state: essayState):
    prompt = f'Based on the following feedbacks, create a summarized feedback \n language quality feedback - {state["language_feedback"]} \n analytical quality feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = llm.invoke(prompt).content

    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback': overall_feedback,
            'avg_score': avg_score}

In [23]:
graph = StateGraph(essayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'essay': 'sample essay here'
}

workflow.invoke(initial_state)